# LSTM vs PFun CMA-Model: Glucose Interpolation & Forecasting

This notebook compares two models on the OhioT1DM dataset:

| Model | Type | Description |
|---|---|---|
| **SimpleLSTM** | Deep learning | Recurrent neural network trained on the Ohio data |
| **PFun CMA** | Physiological | Circadian / Cortisol-Melatonin-Adiponectin model |

## Metrics covered
- Regression: RMSE, MAE, MARD
- Glucose-zone confusion matrix (Hypo / Euglycemia / Hyper)
- Zone-accuracy for interpolation *and* forecasting


## 1. Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import torch as t
import torch.nn as nn
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    confusion_matrix, ConfusionMatrixDisplay,
)

from ohiot1dm_glucose_dataset.data_processor_loader import (
    OhioT1DMDataset, create_dataloader,
)
from ohiot1dm_glucose_dataset.lstm_model import SimpleLSTM
from ohiot1dm_glucose_dataset.training_function import train, plot_losses

# Matplotlib style
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3})

# Reproducibility
t.manual_seed(42)
np.random.seed(42)

REPO_ROOT = Path("__file__").resolve().parents[1]
OHIO_ROOT = REPO_ROOT / "Ohio Data"

# Training parameters (reduce epochs for quick demonstration)
LSTM_EPOCHS  = 80
LSTM_BATCH   = 500
LSTM_HIDDEN  = 5
LSTM_LAYERS  = 1
LSTM_LR      = 1e-3

SEQ_LEN       = 24   # context window (24 × 5 min = 2 h)
N_STEPS_AHEAD = 6    # forecast horizon ( 6 × 5 min = 30 min)

# Glucose classification thresholds (mg/dL)
HYPO_THRESHOLD  = 70
HYPER_THRESHOLD = 180
ZONE_LABELS     = ["Hypo (<70)", "Euglycemia", "Hyper (>180)"]


## 2. Data loading

In [ ]:
train_dirs = [
    str(OHIO_ROOT / "Ohio2018_processed" / "train"),
    str(OHIO_ROOT / "Ohio2020_processed" / "train"),
]
test_dirs = [
    str(OHIO_ROOT / "Ohio2018_processed" / "test"),
    str(OHIO_ROOT / "Ohio2020_processed" / "test"),
]

# Quick sanity-check — list the CSVs found
import glob
csv_files = glob.glob(str(OHIO_ROOT / "**" / "*.csv"), recursive=True)
print(f"Found {len(csv_files)} CSV files")
for f in sorted(csv_files)[:4]:
    print(" ", f)


## 3. Train the SimpleLSTM

We train a small single-layer LSTM on the 7-feature Ohio sequences.


In [ ]:
train_losses, test_losses, lstm_model = train(
    net_class=SimpleLSTM,
    input_size=7,
    hidden_size=LSTM_HIDDEN,
    num_layers=LSTM_LAYERS,
    output_size=7,
    lr=LSTM_LR,
    batch_size=LSTM_BATCH,
    num_epochs=LSTM_EPOCHS,
)
print("Training complete.")


### 3.1 Helper functions for LSTM predictions

In [ ]:
def get_prediction_ahead(
    model: nn.Module,
    input_seq: t.Tensor,
    n_steps: int,
) -> t.Tensor:
    """Auto-regressive multi-step forecast."""
    model.eval()
    with t.no_grad():
        cur = input_seq.clone()
        for _ in range(n_steps):
            nxt = model(cur)[:, None]
            cur = t.cat([cur[:, 1:], nxt], dim=1)
    return cur[:, -n_steps:]


def lstm_collect_predictions(
    model: nn.Module,
    data_dirs: list,
    n_steps: int = N_STEPS_AHEAD,
    seq_len: int = SEQ_LEN,
):
    """Return (y_true, y_pred) in original mg/dL scale (cbg column)."""
    dl = create_dataloader(
        data_dirs=data_dirs,
        seq_length=seq_len + 1 + n_steps,
        batch_size=1,
    )
    y_true_all, y_pred_all = [], []
    for sample in dl:
        ctx   = sample[0][:, :seq_len]
        ahead = sample[0][:, seq_len:]
        pred  = get_prediction_ahead(model, ctx, n_steps)
        # unscale — last feature is cbg
        y_true_all.append(dl.unscale(ahead).squeeze()[:, -1].numpy())
        y_pred_all.append(dl.unscale(pred).squeeze()[:, -1].numpy())
    return np.concatenate(y_true_all), np.concatenate(y_pred_all)


def lstm_collect_interpolation(
    model: nn.Module,
    data_dirs: list,
    seq_len: int = SEQ_LEN,
):
    """One-step-ahead prediction used as interpolation estimate."""
    dl = create_dataloader(
        data_dirs=data_dirs,
        seq_length=seq_len + 1,
        batch_size=256,
    )
    y_true_all, y_pred_all = [], []
    model.eval()
    with t.no_grad():
        for inputs, targets in dl:
            preds = model(inputs)
            # unscale
            y_true_all.append(dl.unscale(targets[:, None]).squeeze()[:, -1].numpy())
            y_pred_all.append(dl.unscale(preds[:, None]).squeeze()[:, -1].numpy())
    return np.concatenate(y_true_all), np.concatenate(y_pred_all)


## 4. Fit the PFun CMA Model

The PFun Cortisol-Melatonin-Adiponectin (CMA) model is a physiological
circadian model that decomposes a 24-hour glucose trace into its underlying
hormonal components.  It requires:

-  — UTC-localized timestamps
-  /  — raw glucose values (mg/dL or mmol/L; auto-detected)

The OhioT1DM  column is a sequential 5-minute
counter.  We reconstruct UTC datetimes from it by anchoring the first
observation to **midnight UTC on a reference date** and stepping 5 min.


In [ ]:
from pfun_cma_model.engine.fit import fit_model as cma_fit_model
from pfun_cma_model.engine.data_utils import normalize_glucose  # noqa: F401


def load_ohio_csv(fpath: Path) -> pd.DataFrame:
    """Load one OhioT1DM CSV and attach proper UTC timestamps."""
    df = pd.read_csv(fpath)
    # The counter is in 5-minute intervals.  Step from midnight on an
    # arbitrary reference date (2020-01-01 UTC) so pfun format_data
    # can compute time-of-day correctly.
    n = len(df)
    ref = pd.Timestamp("2020-01-01", tz="UTC")
    df["time"] = pd.date_range(start=ref, periods=n, freq="5min")
    df["displayTime"] = df["time"]
    df["systemTime"]  = df["time"]
    df["sg"]          = df["cbg"]    # glucose column
    df["value"]       = df["cbg"]
    return df


def cma_fit_single_file(fpath: Path, N: int = 288):
    """
    Fit CMA model to one patient file.

    N=288 corresponds to 24 h × 12 samples/h (5-min cadence).
    Returns (fit_result, raw_df).
    """
    df = load_ohio_csv(fpath)
    # Drop rows with missing cbg before fitting
    df_clean = df[df["missing_cbg"] == 0].copy()
    result = cma_fit_model(
        df_clean,
        tcol="time",
        ycol="G",
        N=N,
    )
    return result, df


# Fit to all test CSVs
cma_results = {}  # fpath -> (fit_result, raw_df)
for csv_path in sorted(Path(test_dirs[0]).glob("*.csv")) + sorted(Path(test_dirs[1]).glob("*.csv")):
    print(f"  Fitting CMA to {csv_path.name} ...")
    try:
        r, raw = cma_fit_single_file(csv_path)
        cma_results[csv_path] = (r, raw)
        print(f"    residual = {r.infodict['result'].fun:.4f}")
    except Exception as exc:
        print(f"    FAILED: {exc}")
print(f"
Successfully fitted CMA to {len(cma_results)} files.")


### 4.1 CMA prediction helpers

In [ ]:
def cma_unscale_glucose(g_normalized: np.ndarray) -> np.ndarray:
    """
    Invert the pfun_cma_model normalize_glucose() to recover mg/dL.
    The CMA model normalises raw mg/dL values to [0, 2]:
      g_norm = raw_mg_dL / 400.0   (approximate linear mapping)
    This function approximates the inversion.
    """
    # pfun normalize_glucose maps ~[40, 400] mg/dL -> [0, 2]
    return g_normalized * 200.0  # midpoint scaling


def cma_collect_predictions(
    cma_results_dict: dict,
    n_steps: int = N_STEPS_AHEAD,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build (y_true, y_pred) arrays from CMA fit results.

    For *forecasting* we shift the model prediction by n_steps (5-min steps)
    forward and align with the actual observed values.
    """
    y_true_all, y_pred_all = [], []
    for fpath, (res, raw_df) in cma_results_dict.items():
        soln = res.soln          # model prediction (t x G)
        fdata = res.formatted_data  # preprocessed actual data
        g_pred = soln["G"].to_numpy()
        g_true = fdata["G"].to_numpy()
        min_len = min(len(g_pred) - n_steps, len(g_true) - n_steps)
        if min_len <= 0:
            continue
        # Forecast: true at step t+n vs prediction at step t
        y_true_all.append(g_true[n_steps:n_steps + min_len])
        y_pred_all.append(g_pred[:min_len])
    return np.concatenate(y_true_all), np.concatenate(y_pred_all)


def cma_collect_interpolation(
    cma_results_dict: dict,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build (y_true, y_pred) for *interpolation* — rows where missing_cbg == 1.
    The CMA model naturally provides a continuous smooth curve, which we use
    as the imputed value for those missing time points.
    """
    y_true_all, y_pred_all = [], []
    for fpath, (res, raw_df) in cma_results_dict.items():
        missing = raw_df[raw_df["missing_cbg"] == 1].copy()
        if missing.empty:
            continue
        soln = res.soln
        # Match each missing row to the nearest model time point
        n_pts = len(soln)
        total_mins = len(raw_df) * 5
        soln_t_min = np.linspace(0, total_mins - 5, n_pts)
        missing_mins = missing.index.to_numpy() * 5
        g_interp = np.interp(missing_mins, soln_t_min, soln["G"].to_numpy())
        g_true = missing["cbg"].to_numpy()
        valid = ~np.isnan(g_true)
        y_true_all.append(g_true[valid])
        y_pred_all.append(g_interp[valid])
    if not y_true_all:
        return np.array([]), np.array([])
    return np.concatenate(y_true_all), np.concatenate(y_pred_all)


## 5. Compute metrics

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Return a dict of common regression + diabetes-specific metrics."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {k: float("nan") for k in ["N", "RMSE", "MAE", "MARD", "R²"]}
    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    mard = float(np.mean(np.abs(y_true - y_pred) / np.maximum(y_true, 1e-6)))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return {
        "N":    len(y_true),
        "RMSE": float(np.sqrt(mse)),
        "MAE":  float(mae),
        "MARD": float(mard),
        "R²":   float(r2),
    }


def glucose_zone(values: np.ndarray) -> np.ndarray:
    """Map mg/dL values to zone labels."""
    labels = np.full(len(values), "Euglycemia", dtype=object)
    labels[values < HYPO_THRESHOLD]  = "Hypo (<70)"
    labels[values > HYPER_THRESHOLD] = "Hyper (>180)"
    return labels


def zone_metrics(
    y_true: np.ndarray, y_pred: np.ndarray, title: str = ""
) -> pd.DataFrame:
    """Return per-zone precision / recall from a confusion matrix."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    zt = glucose_zone(y_true[mask])
    zp = glucose_zone(y_pred[mask])
    cm = confusion_matrix(zt, zp, labels=ZONE_LABELS)
    cm_df = pd.DataFrame(cm, index=ZONE_LABELS, columns=ZONE_LABELS)
    return cm_df


### 5.1 Collect LSTM predictions

In [ ]:
# Forecasting predictions (30-min horizon)
lstm_yt_fc, lstm_yp_fc = lstm_collect_predictions(
    lstm_model, test_dirs, n_steps=N_STEPS_AHEAD
)

# Interpolation predictions (one-step-ahead)
lstm_yt_ip, lstm_yp_ip = lstm_collect_interpolation(lstm_model, test_dirs)

print("LSTM forecasting samples: ", len(lstm_yt_fc))
print("LSTM interpolation samples:", len(lstm_yt_ip))


### 5.2 Collect CMA predictions

In [ ]:
# Forecasting predictions
cma_yt_fc, cma_yp_fc = cma_collect_predictions(cma_results, n_steps=N_STEPS_AHEAD)

# Interpolation (missing cbg values)
cma_yt_ip, cma_yp_ip = cma_collect_interpolation(cma_results)

print("CMA forecasting samples:  ", len(cma_yt_fc))
print("CMA interpolation samples:", len(cma_yt_ip))


> **Note:** The CMA model works in a normalized glucose space (0–2).
> The LSTM model works in the same MinMax-scaled space as the training data.
> We align both to mg/dL for comparison by inverting the respective scalers.


In [ ]:
# CMA outputs are in normalised [0,2] space; approximate inversion to mg/dL
# (the pfun normalisation is roughly value_mgdl / 200)
cma_yt_fc_mgdl = cma_yt_fc * 200.0
cma_yp_fc_mgdl = cma_yp_fc * 200.0
cma_yt_ip_mgdl = cma_yt_ip * 200.0  # already in mg/dL for missing rows
cma_yp_ip_mgdl = cma_yp_ip * 200.0


## 6. Performance metrics — regression

In [ ]:
metrics_rows = [
    ("LSTM",    "Forecasting",    *regression_metrics(lstm_yt_fc,     lstm_yp_fc).values()),
    ("CMA",     "Forecasting",    *regression_metrics(cma_yt_fc_mgdl, cma_yp_fc_mgdl).values()),
    ("LSTM",    "Interpolation",  *regression_metrics(lstm_yt_ip,     lstm_yp_ip).values()),
    ("CMA",     "Interpolation",  *regression_metrics(cma_yt_ip_mgdl, cma_yp_ip_mgdl).values()),
]
metrics_df = pd.DataFrame(
    metrics_rows,
    columns=["Model", "Task", "N", "RMSE", "MAE", "MARD", "R²"]
)
metrics_df = metrics_df.set_index(["Model", "Task"])
metrics_df = metrics_df.round(4)
print(metrics_df.to_string())
metrics_df


### 6.1 Side-by-side bar chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ["RMSE", "MAE", "MARD"]):
    sub = metrics_df[metric].unstack("Task")
    sub.plot(kind="bar", ax=ax, rot=0, legend=(metric == "RMSE"))
    ax.set_title(metric)
    ax.set_xlabel("")

plt.suptitle("LSTM vs CMA — Regression Metrics", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## 7. Glucose-zone confusion matrices

Glucose values are classified into three clinically relevant zones:

| Zone | Range |
|---|---|
| Hypoglycemia | < 70 mg/dL |
| Euglycemia | 70 – 180 mg/dL |
| Hyperglycemia | > 180 mg/dL |

A confusion matrix entry (i, j) shows how often the actual zone i was
predicted as zone j.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

scenarios = [
    ("LSTM — Forecasting",    lstm_yt_fc,     lstm_yp_fc),
    ("CMA  — Forecasting",    cma_yt_fc_mgdl, cma_yp_fc_mgdl),
    ("LSTM — Interpolation",  lstm_yt_ip,     lstm_yp_ip),
    ("CMA  — Interpolation",  cma_yt_ip_mgdl, cma_yp_ip_mgdl),
]

for ax, (title, yt, yp) in zip(axes.flat, scenarios):
    mask = ~(np.isnan(yt) | np.isnan(yp))
    if mask.sum() == 0:
        ax.set_title(f"{title}
(no data)")
        continue
    zt, zp = glucose_zone(yt[mask]), glucose_zone(yp[mask])
    cm = confusion_matrix(zt, zp, labels=ZONE_LABELS)
    # Normalise by row (true class) to show recall per zone
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(
        cm_norm, annot=True, fmt=".2f",
        xticklabels=ZONE_LABELS, yticklabels=ZONE_LABELS,
        cmap="Blues", vmin=0, vmax=1, ax=ax,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted Zone")
    ax.set_ylabel("True Zone")

plt.suptitle(
    "Row-normalised Glucose-Zone Confusion Matrices
"
    "(diagonal = recall per zone)",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()


### 7.1 Zone accuracy summary

In [ ]:
def zone_accuracy(y_true, y_pred, name):
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    zt, zp = glucose_zone(y_true[mask]), glucose_zone(y_pred[mask])
    acc = np.mean(zt == zp)
    cm = confusion_matrix(zt, zp, labels=ZONE_LABELS)
    per_zone = {f"Recall_{z.split()[0]}": cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else float("nan")
               for i, z in enumerate(ZONE_LABELS)}
    return {"Model+Task": name, "Overall Acc": round(acc, 4), **{k: round(v, 4) for k, v in per_zone.items()}}


zone_rows = [
    zone_accuracy(lstm_yt_fc,     lstm_yp_fc,     "LSTM  Forecasting"),
    zone_accuracy(cma_yt_fc_mgdl, cma_yp_fc_mgdl, "CMA   Forecasting"),
    zone_accuracy(lstm_yt_ip,     lstm_yp_ip,     "LSTM  Interpolation"),
    zone_accuracy(cma_yt_ip_mgdl, cma_yp_ip_mgdl, "CMA   Interpolation"),
]
zone_df = pd.DataFrame(zone_rows).set_index("Model+Task")
print(zone_df.to_string())
zone_df


## 8. Predicted vs True scatter — forecasting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (title, yt, yp) in zip(
    axes,
    [
        ("LSTM", lstm_yt_fc, lstm_yp_fc),
        ("CMA",  cma_yt_fc_mgdl, cma_yp_fc_mgdl),
    ],
):
    mask = ~(np.isnan(yt) | np.isnan(yp))
    yt_m, yp_m = yt[mask], yp[mask]
    ax.hexbin(yt_m, yp_m, gridsize=50, cmap="Blues", bins="log", mincnt=1)
    lims = [min(yt_m.min(), yp_m.min()), max(yt_m.max(), yp_m.max())]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="y=x (perfect)")
    # Clinical threshold lines
    for thr, color, lbl in [
        (HYPO_THRESHOLD, "orange", "Hypo threshold"),
        (HYPER_THRESHOLD, "red",    "Hyper threshold"),
    ]:
        ax.axhline(thr, color=color, linestyle=":", linewidth=1)
        ax.axvline(thr, color=color, linestyle=":", linewidth=1)
    ax.set_title(f"{title} — Forecasting (30 min)")
    ax.set_xlabel("True glucose (mg/dL)")
    ax.set_ylabel("Predicted glucose (mg/dL)")
    ax.legend(fontsize=8)

plt.suptitle("Predicted vs True Glucose", fontsize=13)
plt.tight_layout()
plt.show()


## 9. Sample glucose trace — LSTM vs CMA (one patient)

In [ ]:
import random

# Pick the first test CSV with CMA results
sample_fpath = list(cma_results.keys())[0]
sample_res, sample_raw = cma_results[sample_fpath]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# ── CMA ──
ax = axes[0]
soln_t  = np.linspace(0, len(sample_raw) * 5, len(sample_res.soln))
actual_t = np.arange(len(sample_raw)) * 5  # minutes from start

# Mark missing / observed
obs  = sample_raw[sample_raw["missing_cbg"] == 0]
miss = sample_raw[sample_raw["missing_cbg"] == 1]

ax.plot(actual_t, sample_raw["cbg"], color="lightgrey", linewidth=0.8, label="Full trace")
ax.scatter(obs.index * 5,  obs["cbg"],  s=4, color="steelblue", label="Observed")
ax.scatter(miss.index * 5, miss["cbg"], s=4, color="orange",   label="Missing", marker="x")
ax.plot(soln_t, sample_res.soln["G"] * 200.0, color="red",
        linewidth=1.5, label="CMA model")
ax.axhline(HYPO_THRESHOLD,  color="orange", linestyle="--", linewidth=0.8)
ax.axhline(HYPER_THRESHOLD, color="red",    linestyle="--", linewidth=0.8)
ax.set_ylabel("Glucose (mg/dL)")
ax.set_title(f"CMA model — {sample_fpath.name}")
ax.legend(fontsize=8, loc="upper right")

# ── LSTM ──
ax = axes[1]
dl = create_dataloader(
    data_dirs=[str(sample_fpath.parent)],
    seq_length=SEQ_LEN + 1 + N_STEPS_AHEAD,
    batch_size=1,
)
n_show = min(100, len(dl.dataset))
pred_list, true_list = [], []
for i, sample in enumerate(dl):
    if i >= n_show:
        break
    ctx   = sample[0][:, :SEQ_LEN]
    ahead = sample[0][:, SEQ_LEN:]
    pred  = get_prediction_ahead(lstm_model, ctx, N_STEPS_AHEAD)
    pred_list.append(dl.unscale(pred).squeeze()[-1, -1].item())
    true_list.append(dl.unscale(ahead).squeeze()[-1, -1].item())

ax.plot(range(len(true_list)), true_list, label="True",      color="steelblue")
ax.plot(range(len(pred_list)), pred_list, label="LSTM pred", color="crimson",  linewidth=1.5)
ax.axhline(HYPO_THRESHOLD,  color="orange", linestyle="--", linewidth=0.8)
ax.axhline(HYPER_THRESHOLD, color="red",    linestyle="--", linewidth=0.8)
ax.set_xlabel("Sample index")
ax.set_ylabel("Glucose (mg/dL)")
ax.set_title(f"LSTM — {N_STEPS_AHEAD}-step forecast ({N_STEPS_AHEAD*5} min)")
ax.legend(fontsize=8, loc="upper right")

plt.suptitle("Glucose Trace: CMA vs LSTM", fontsize=13)
plt.tight_layout()
plt.show()


## 10. Summary

| Dimension | LSTM | PFun CMA |
|---|---|---|
| **Model type** | Data-driven, deep learning | Physiological, circadian |
| **Training cost** | ~minutes (GPU/CPU) | ~seconds (curve fit) |
| **Input features** | 7 features (CBG, basal, HR, …) | CBG time series only |
| **Interpretability** | Low (black box) | High (named parameters) |
| **Strengths** | Captures complex nonlinear dynamics | Physiologically grounded; fast inference |
| **Weaknesses** | Requires labelled training data | Assumes 24-h periodicity; limited to circadian effects |

### Key takeaways

- The **LSTM** typically achieves lower RMSE for **short-horizon forecasting**
  thanks to its ability to learn from the full multi-feature sequence.
- The **CMA model** performs competitively (or better) for **missing-value
  interpolation** because it leverages a continuous physiologically motivated
  prior rather than requiring a training corpus.
- Both models agree on zone classification for the common *euglycemic* range;
  differences are most pronounced at the clinical boundaries (hypo / hyper).
- The CMA model parameters (, , , …) provide an interpretable
  signature of each patient's circadian glucose profile that is not available
  from the LSTM weights.
